# 01b · Evaluación RAG++ (Advanced RAG)
Evalúa RAG++ (búsqueda híbrida + reranking) sobre UltraDomain con métricas RAGAS.

**Mejoras sobre Traditional RAG:**
- EnsembleRetriever: ChromaDB vectorial (60%) + BM25 léxico (40%)
- CrossEncoder reranker: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Recupera 20 candidatos → filtra a los 4 mejores

In [6]:
import os
import sys
from dotenv import load_dotenv

sys.path.append(os.path.abspath('../..'))
load_dotenv(os.path.join('../..', '.env'))

True

## 1. Configuración

In [7]:
DOMINIO     = "cs"
N_LIBROS    = None
MAX_Q       = None
SHUFFLE     = False
CARPETA_DB  = "../../chroma_db_RAGPlusPlus"
RESULTS_DIR = "../../results_RAGPlusPlus/"

# Parámetros RAG++
VECTOR_K      = 10    # candidatos del retriever vectorial
BM25_K        = 10    # candidatos del retriever BM25
RERANK_TOP_N  = 4     # chunks finales tras reranking
VECTOR_WEIGHT = 0.6   # peso vectorial en el ensemble
BM25_WEIGHT   = 0.4   # peso BM25 en el ensemble

## 2. Cargar datos

In [8]:
from src.ultradomain import cargar_experimento

libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

📚 Dominio: cs
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   Total Q&A: 100


## 3. Inicializar e indexar RAG++

In [ ]:
import shutil
from src.data_loader import load_and_split_text
from src.baselines.advanced_rag import AdvancedRAG

rag = AdvancedRAG(
    persist_directory=CARPETA_DB,
    vector_k=VECTOR_K,
    bm25_k=BM25_K,
    rerank_top_n=RERANK_TOP_N,
    vector_weight=VECTOR_WEIGHT,
    bm25_weight=BM25_WEIGHT,
)

for libro in libros:
    tmp = f"/tmp/{libro['context_id']}.txt"
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(libro["texto"])
    splits = load_and_split_text(tmp)
    print(f"📖 {libro['titulo']} → {len(splits)} fragmentos")
    rag.index_documents(splits)

print("✅ Indexación completada")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📖 Mastering VBA for Microsoft Office 2013 → 2747 fragmentos
📖 Probability and Statistics for Computer Science → 1658 fragmentos
📖 Introducing Regular Expressions → 320 fragmentos
📖 Introduction to the Theory of Programming Languages → 266 fragmentos
📖 Machine Learning With Spark → 764 fragmentos
📖 Professional Microsoft SQL Server 2008 Programming → 2461 fragmentos
📖 Guide to Java → 845 fragmentos
📖 Joe Celko's SQL Programming Style → 425 fragmentos
📖 Linux Kernel Networking → 1849 fragmentos
📖 Modern Optimization With R → 489 fragmentos
✅ Indexación completada


## 4. Ejecutar evaluación

In [10]:
from src.evaluation.experiment import run_experiment

result = await run_experiment(
    rag_type="advanced",   # usa el mismo adapter que traditional RAG
    rag_object=rag,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    save_path=RESULTS_DIR,
)

/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/instructor/providers/gemini/client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


📚 Dominio: cs
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   Total Q&A: 100

🔍 Evaluando ADVANCED | cs | 100 preguntas
────────────────────────────────────────────────────────────
  [1/100] How does Spark Streaming enable real-time data processing?...
  [2/100] What does the book suggest about the use of histograms in data analysi...
 

## 5. Inspeccionar respuestas individuales

In [ ]:
for r in result.qa_results:
    print(f"❓ {r.question}")
    print(f"✅ GT:  {r.ground_truth[:150]}...")
    print(f"🤖 RAG: {r.answer[:150]}...")
    print(f"⏱️  {r.latency_s}s\n")